[![Labellerr](https://storage.googleapis.com/labellerr-cdn/%200%20Labellerr%20template/notebook.webp)](https://www.labellerr.com)

# **Smart Traffic Monitoring using AI**

---

[![labellerr](https://img.shields.io/badge/Labellerr-BLOG-black.svg)](https://www.labellerr.com/blog/<BLOG_NAME>)
[![Youtube](https://img.shields.io/badge/Labellerr-YouTube-b31b1b.svg)](https://www.youtube.com/@Labellerr)
[![Github](https://img.shields.io/badge/Labellerr-GitHub-green.svg)](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)

## Overview
This notebook demonstrates an end-to-end computer vision pipeline for **Intelligent Traffic Analysis** using a YOLO-based segmentation and tracking model. The workflow covers high-precision vehicle detection, multi-object tracking, and real-time speed estimation through custom pixel-to-meter calibration.

By leveraging retina-level segmentation masks, the system generates cumulative **"signal" heatmaps** that visualize road usage density and automatically identify congestion bottlenecks based on vehicle velocity patterns.

---

##  Core Capabilities

- **High-Precision Vehicle Detection** using YOLO segmentation  
- **Multi-Object Tracking (MOT)** for consistent vehicle ID assignment  
- **Real-Time Speed Estimation** via pixel-to-meter calibration  
- **Density Heatmap Generation** for traffic flow visualization  
- **Automatic Congestion Detection** based on velocity thresholds  

---

##  Real-World Applications

### 1. Smart City Traffic Management
Optimize traffic light timings and lane allocations based on real-time vehicle density and traffic flow rates.

### 2. Automated Speed Enforcement
Detect and log vehicles exceeding local speed limits without requiring manual radar traps.

### 3. Infrastructure Planning
Use long-term cumulative heatmap signals to identify road segments prone to congestion, enabling data-driven urban redesign.

### 4. Safety Monitoring
Automatically detect sudden stops or slow-moving traffic that may indicate accidents or road hazards.

### 5. Logistics and Fleet Analysis
Analyze route efficiency and bottleneck locations to improve commercial delivery performance and reduce transit times.

## Annotate your Custom dataset using Labellerr

 ***1. Visit the [Labellerr](https://www.labellerr.com/?utm_source=githubY&utm_medium=social&utm_campaign=github_clicks) website and click **“Sign Up”**.*** 

 ***2. After signing in, create your workspace by entering a unique name.***

 ***3. Navigate to your workspace’s API keys page (e.g., `https://<your-workspace>.labellerr.com/workspace/api-keys`) to generate your **API Key** and **API Secret**.***

 ***4. Store the credentials securely, and then use them to initialise the SDK or API client with `api_key`, `api_secret`.*** 



In [1]:
!git clone https://github.com/Labellerr/yolo_finetune_utils.git

Cloning into 'yolo_finetune_utils'...


## Random Frame Extraction from Video

Extracts a fixed number of high-quality frames from one or more videos to create an image dataset for annotation and training.

### Purpose
- Convert raw manufacturing videos into individual image frames  
- Perform random sampling to avoid frame bias  
- Prepare data for annotation and YOLO training  


In [2]:
from yolo_finetune_utils.frame_extractor import extract_random_frames

extract_random_frames(
    paths=['Untitled design (1).mp4'],
    total_images=25,
    out_dir="dataset_frames",
    jpg_quality=100,
    seed=42
)

[✓] Extracted 25 frames to folder: dataset_frames


## Download Annotations from Labellerr

After completing data labeling on the **Labellerr** platform, export the annotations in **COCO JSON format**.

Download the COCO JSON file from the Labellerr website and upload it into this project workspace to use it for further dataset preparation and training.

This COCO JSON file will be used in the next steps for:
- Frame–annotation alignment
- COCO → YOLO format conversion
- Model training and evaluation


# COCO to YOLO Format Conversion

Converts COCO-style segmentation annotations to YOLO segmentation dataset format.  
- Requires: `annotation.json` and images in `frames_output` directory.
- Output: Generated YOLO dataset folder.
- Parameters: allows train/val split, shuffling, and verbose mode.


In [2]:
from yolo_finetune_utils.coco_yolo_converter.seg_converter import coco_to_yolo_converter

coco_to_yolo_converter(
    json_path="export-#1b0f0e2b-f3c7-49c5-9772-62cdde2680fe.json",
    images_dir="dataset_frames",
    output_dir="yolo_dataset",
    use_split=True,
    train_ratio=0.7,
    val_ratio=0.2,
    test_ratio=0.1,
    shuffle=True,
    verbose=True
)

Conversion complete. Stats: {'train': 17, 'val': 5, 'test': 3}


{'stats': {'train': 17, 'val': 5, 'test': 3}, 'output_dir': 'yolo_dataset'}

In [4]:
!pip install -U ultralytics

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   -------------------------- ------------- 0.8/1.2 MB 7.5 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 7.9 MB/s  0:00:00
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.4.16
    Uninstalling ultralytics-8.4.16:
      Successfully uninstalled ultralytics-8.4.16


# Load and Train YOLO Segmentation Model

Loads the YOLO segmentation model and trains it using the converted YOLO dataset.
- Data: Path to YOLO-style `data.yaml`
- Parameters: epochs, image size, batch size, device, dataloader workers, experiment name.


In [ ]:
from ultralytics import YOLO

# 1. Load a pretrained YOLO11 Segmentation model
# Using 'yolo11x-seg.pt' for efficiency on your laptop
model = YOLO("yolo11x-seg.pt") 

# 2. Train the model
results = model.train(
    data="yolo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    workers=0,                # Keeps it stable on Windows
    device='cpu',             # Switch to 0 once CUDA is fixed
    project="Traffic_Flow_Seg",
    name="seg_v1"
)

# YOLO Segmentation + Tracking Inference Pipeline (Logic Explanation)

This code block performs **GPU-based vehicle segmentation, tracking, and video inference** using a trained YOLO segmentation model. Below is a step-by-step explanation of the logic behind each section.


In [ ]:
import torch
from ultralytics import YOLO

# 1. Clear GPU memory
torch.cuda.empty_cache()

model_path = '/kaggle/working/runs/segment/Traffic_Seg_Kaggle/seg_run_02/weights/best.pt'
model = YOLO(model_path)

# 2. Start the Generator
results = model.track(
    source='/kaggle/input/datasets/aaryanaggarwal5040/raw-video/Untitled design (1).mp4', 
    persist=True, 
    retina_masks=True, 
    conf=0.15,
    iou=0.4,
    show_boxes=False,     
    show_conf=False,      
    save=True,           
    stream=True,          # This makes 'results' a generator
    project="Inference_Output",
    name="precise_traffic_fixed"
)

# 3. CONSUME THE GENERATOR (This starts the actual processing)
# Without this loop, the code just defines the task and stops
for r in results:
    pass 

print("Processing finished and video saved.")

# Interactive Pixel-to-Meter Calibration Tool

This script implements an **interactive calibration tool** that converts image pixel distances into real-world physical units (meters).  

This step is **mandatory for accurate vehicle speed estimation**, because object tracking alone provides movement in pixels — not meters.

Without calibration, speed calculations would be meaningless in real-world units.

---

# Core Idea: The “Digital Ruler” Concept

The calibration logic is based on a known physical constant:

> **Average length of a passenger car ≈ 4.5 meters**

If we measure how many **pixels** represent a 4.5-meter vehicle in the video, we can compute a scaling factor:

\[
\textbf{Meters Per Pixel (MPP)} = \frac{\text{Real-World Length (meters)}}{\text{Measured Pixel Length}}
\]

This converts every pixel movement into meters.

Think of it as placing a **digital ruler** inside your video.


In [1]:
import cv2
import random
import numpy as np

# --- CONFIGURATION ---
VIDEO_PATH = "Untitled design (1).mp4"
REAL_CAR_LENGTH = 4.5  # Meters (Standard UK/EU average)

points = []

def click_event(event, x, y, flags, params):
    """Handles mouse clicks to capture two points."""
    if event == cv2.EVENT_LBUTTONDOWN:
        points.append((x, y))
        # Draw a circle at the click point
        cv2.circle(img, (x, y), 5, (0, 0, 255), -1)
        cv2.imshow("Calibration: Click Front then Back of a Car", img)
        
        if len(points) == 2:
            # Calculate Euclidean distance in pixels
            p1, p2 = points[0], points[1]
            pixel_dist = np.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)
            
            # Calculate Meters Per Pixel
            mpp = REAL_CAR_LENGTH / pixel_dist
            
            print(f"\n--- CALIBRATION RESULTS ---")
            print(f"Point 1: {p1} | Point 2: {p2}")
            print(f"Distance in Pixels: {pixel_dist:.2f}")
            print(f"Meters Per Pixel (MPP): {mpp:.6f}")
            print(f"---------------------------\n")
            print("Press any key to close and use these values in your main script.")

# 1. Load Video and pick a random frame
cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
random_frame_idx = random.randint(0, total_frames - 1)

cap.set(cv2.CAP_PROP_POS_FRAMES, random_frame_idx)
success, img = cap.read()
cap.release()

if not success:
    print("Error: Could not read video.")
else:
    # 2. Open Window for clicking
    cv2.imshow("Calibration: Click Front then Back of a Car", img)
    cv2.setMouseCallback("Calibration: Click Front then Back of a Car", click_event)
    
    print("INSTRUCTIONS:")
    print("1. Find a standard car in the frame.")
    print("2. Click exactly on the front bumper.")
    print("3. Click exactly on the rear bumper.")
    
    cv2.waitKey(0)
    cv2.destroyAllWindows()

INSTRUCTIONS:
1. Find a standard car in the frame.
2. Click exactly on the front bumper.
3. Click exactly on the rear bumper.

--- CALIBRATION RESULTS ---
Point 1: (715, 257) | Point 2: (648, 243)
Distance in Pixels: 68.45
Meters Per Pixel (MPP): 0.065744
---------------------------

Press any key to close and use these values in your main script.


# Intelligent Traffic Analysis System  
### (YOLO Segmentation + Tracking + Speed Estimation + Heatmaps)

This system implements a **physics-aware traffic analytics pipeline** that transforms raw video into quantitative intelligence by combining deep learning, geometry, and motion physics.

---

## 1️⃣ System Architecture

The pipeline follows a structured processing sequence:

```
Input Video 
   → YOLO Segmentation 
      → Object Tracking 
         → Trajectory Extraction 
            → Speed Calculation 
               → Heatmap Accumulation 
                  → Visual Output
```

Each stage progressively converts raw pixel data into interpretable traffic intelligence.

---

## 2️⃣ Calibration: Pixels to Meters

To calculate real-world speed, we define a **Meters Per Pixel (MPP)** constant:

\[
MPP = \frac{\text{Known Object Length (4.5m)}}{\text{Object Length in Pixels (P)}}
\]

This calibration ensures the system remains:

- Camera-height agnostic  
- Resolution-independent  
- Physically interpretable  

---

## 3️⃣ Segmentation & Tracking

### Segmentation (YOLO)

- YOLO predicts precise pixel masks \( M_i \) for each detected vehicle.
- Provides higher spatial accuracy than traditional bounding boxes.
- Enables precise centroid and contour extraction.

### Tracking (BoT-SORT)

- Assigns persistent IDs using:
  - Kalman Filtering
  - IoU matching
- Ensures temporal consistency for trajectory analysis.
- Maintains identity across occlusions and frame drops.

---

## 4️⃣ Physics-Based Speed Estimation

Speed is computed using Euclidean distance over a sliding window of 20 frames to reduce detection noise.

### Pixel Distance

\[
d_p = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}
\]

### Real-World Distance

\[
d_m = d_p \times MPP
\]

### Velocity (km/h)

\[
v = \left(\frac{d_m}{\text{Frames} / \text{FPS}}\right) \times 3.6
\]

Where:

- `FPS` = Frames per second  
- `3.6` converts m/s → km/h  

This approach ensures:

- Noise smoothing  
- Physically consistent measurements  
- Accurate per-vehicle speed estimation  

---

## 5️⃣ Velocity-Weighted Heatmaps

Congestion intensity is modeled inversely proportional to vehicle speed.

### Weight Function

\[
w = 1 + 9(1 - ratio)
\]

Where:

\[
ratio = \frac{\text{vehicle speed}}{\text{speed limit}}
\]

### Logic Interpretation

| Vehicle State | Speed | Heat Contribution |
|---------------|-------|------------------|
| Stopped       | 0 km/h | 10× |
| Slow          | Low    | High |
| Fast          | Near limit | Low |

Bottleneck areas gradually turn **Red** due to accumulated congestion energy.

---

## 6️⃣ Spatial Signal & Exponential Decay

To maintain long-term traffic memory:

\[
H_{new} = H_{old} \times 0.999 + \text{current\_traffic}
\]

### Why Exponential Decay?

- Preserves persistent congestion signatures  
- Gradually fades inactive regions  
- Creates a dynamic yet stable traffic heat signature  

This enables long-duration analysis without storing full historical frames.

---

## 7️⃣ Final Composition

The final frame is generated by:

- Blending original frame and normalized heatmap
- Ratio: **70% original + 30% heatmap**
- Rendering speed labels:
  - Black color
  - Centered at vehicle centroid
  - Optimized for contrast against dynamic backgrounds

---

# Scientific Achievement

This engine bridges:

- Computer Vision  
- Geometry  
- Motion Kinematics  
- Statistical Signal Processing  

## Capabilities

- Speed Monitoring & Automated Enforcement  
- Bottleneck Identification via Congestion Energy  
- Infrastructure Analysis for Smarter Urban Planning  
- Long-Term Traffic Pattern Modeling  

In [ ]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from collections import deque

# 1. SETUP & CALIBRATION
torch.cuda.empty_cache()
MPP = 0.067933  # Your measured Meters Per Pixel
FPS = 60   
SPEED_LIMIT = 50 # Normalizing speed for heatmap intensity
track_history = {}

# Load your specific trained model
model = YOLO('/kaggle/input/datasets/aaryanaggarwal5040/bestmodel/best.pt')

video_input = '/kaggle/input/datasets/aaryanaggarwal5040/raw-video/Untitled design (1).mp4'
cap = cv2.VideoCapture(video_input)
w, h = int(cap.get(3)), int(cap.get(4))
cap.release()

# Use mp4v for Kaggle compatibility
out = cv2.VideoWriter('/kaggle/working/traffic_blackest_text_analysis.mp4', 
                       cv2.VideoWriter_fourcc(*'mp4v'), FPS, (w, h))

# Persistent Heatmap Accumulator
heatmap_acc = np.zeros((h, w), dtype=np.float32)

# 2. ANALYSIS LOOP
results = model.track(source=video_input, persist=True, retina_masks=True, 
                      conf=0.15, stream=True, tracker="botsort.yaml")

for r in results:
    # Use r.plot to get high-precision masks
    annotated_frame = r.plot(boxes=False, labels=False)
    
    if r.boxes.id is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        ids = r.boxes.id.cpu().int().numpy()

        for box, track_id in zip(boxes, ids):
            x1, y1, x2, y2 = box
            cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)

            # --- SPEED CALCULATION ---
            if track_id not in track_history:
                track_history[track_id] = deque(maxlen=20)
            track_history[track_id].append((cx, cy))

            current_speed = 0
            if len(track_history[track_id]) > 1:
                p_old, p_new = track_history[track_id][0], track_history[track_id][-1]
                dist = np.sqrt((p_new[0]-p_old[0])**2 + (p_new[1]-p_old[1])**2) * MPP
                time = len(track_history[track_id]) / FPS
                current_speed = (dist / time) * 3.6

            # --- SMOOTH HEATMAP INTENSITY ---
            # Speed 0 = Weight 10 (Red) | Speed 60+ = Weight 1 (Blue/Green)
            speed_ratio = np.clip(current_speed / SPEED_LIMIT, 0, 1)
            heat_weight = 1.0 + (9.0 * (1.0 - speed_ratio)) 
            cv2.circle(heatmap_acc, (cx, cy), 35, heat_weight, -1)

            # --- FIXED BLACK TEXT DISPLAY ---
            # (0, 0, 0) is the BGR code for Black
            cv2.putText(annotated_frame, f"{int(current_speed)}km/h", (cx, cy-25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

    # 3. RENDER PERSISTENT SIGNAL
    heatmap_acc *= 0.999 # Slow decay for permanent road 'signal'
    heatmap_norm = cv2.normalize(heatmap_acc, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    heatmap_color = cv2.applyColorMap(heatmap_norm, cv2.COLORMAP_JET)
    
    # Overlay heatmap on frame
    final_output = cv2.addWeighted(annotated_frame, 0.7, heatmap_color, 0.3, 0)
    out.write(final_output)

out.release()
print("Process Complete. The final video has been saved with black speed text.")

---

## 👨‍💻 About Labellerr's Hands-On Learning in Computer Vision

Thank you for exploring this **Labellerr Hands-On Computer Vision Cookbook**! We hope this notebook helped you learn, prototype, and accelerate your vision projects.  
Labellerr provides ready-to-run Jupyter/Colab notebooks for the latest models and real-world use cases in computer vision, AI agents, and data annotation.

---
## 🧑‍🔬 Check Our Popular Youtube Videos

Whether you're a beginner or a practitioner, our hands-on training videos are perfect for learning custom model building, computer vision techniques, and applied AI:

- [How to Fine-Tune YOLO on Custom Dataset](https://www.youtube.com/watch?v=pBLWOe01QXU)  
  Step-by-step guide to fine-tuning YOLO for real-world use—environment setup, annotation, training, validation, and inference.
- [Build a Real-Time Intrusion Detection System with YOLO](https://www.youtube.com/watch?v=kwQeokYDVcE)  
  Create an AI-powered system to detect intruders in real time using YOLO and computer vision.
- [Finding Athlete Speed Using YOLO](https://www.youtube.com/watch?v=txW0CQe_pw0)  
  Estimate real-time speed of athletes for sports analytics.
- [Object Counting Using AI](https://www.youtube.com/watch?v=smsjBBQcIUQ)  
  Learn dataset curation, annotation, and training for robust object counting AI applications.
---

## 🎦 Popular Labellerr YouTube Videos

Level up your skills and see video walkthroughs of these tools and notebooks on the  
[Labellerr YouTube Channel](https://www.youtube.com/@Labellerr/videos):

- [How I Fixed My Biggest Annotation Nightmare with Labellerr](https://www.youtube.com/watch?v=hlcFdiuz_HI) – Solving complex annotation for ML engineers.
- [Explore Your Dataset with Labellerr's AI](https://www.youtube.com/watch?v=LdbRXYWVyN0) – Auto-tagging, object counting, image descriptions, and dataset exploration.
- [Boost AI Image Annotation 10X with Labellerr's CLIP Mode](https://www.youtube.com/watch?v=pY_o4EvYMz8) – Refine annotations with precision using CLIP mode.
- [Boost Data Annotation Accuracy and Efficiency with Active Learning](https://www.youtube.com/watch?v=lAYu-ewIhTE) – Speed up your annotation workflow using Active Learning.

> 👉 **Subscribe** for Labellerr's deep learning, annotation, and AI tutorials, or watch videos directly alongside notebooks!

---

## 🤝 Stay Connected

- **Website:** [https://www.labellerr.com/](https://www.labellerr.com/)
- **Blog:** [https://www.labellerr.com/blog/](https://www.labellerr.com/blog/)
- **GitHub:** [Labellerr/Hands-On-Learning-in-Computer-Vision](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)
- **LinkedIn:** [Labellerr](https://in.linkedin.com/company/labellerr)
- **Twitter/X:** [@Labellerr1](https://x.com/Labellerr1)

*Happy learning and building with Labellerr!*
